In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta
import numpy as np

# --- 1. Configuration & Geography ---

EGYPT_GEOGRAPHY = {
    "Cairo":      ["Nasr City", "Maadi", "Heliopolis", "Downtown Cairo", "Zamalek", "Shubra"],
    "Giza":       ["Dokki", "Mohandessin", "Haram", "6th of October City", "Faisal", "Imbaba"],
    "Alexandria": ["Smouha", "Miami", "Sidi Gaber", "Downtown Alexandria", "Montazah", "El Mandara"],
    "Qalyubia":   ["Banha", "Qalyub", "Shubra El Kheima", "Khusus"],
    "Gharbia":    ["Tanta", "Kafr El Zayat", "Mahalla", "El Santa"],
    "Dakahlia":   ["Mansoura", "Mit Ghamr", "Talkha", "El Gamalia"],
    "Sharqia":    ["Zagazig", "10th of Ramadan City", "Abu Hammad", "Belbeis"],
    "Aswan":      ["Aswan City", "Kom Ombo", "Edfu", "Daraw"],
    "Luxor":      ["Luxor City", "Karnak", "Armant", "Esna"],
    "Red Sea":    ["Hurghada", "El Gouna", "Marsa Alam", "Ras Gharib"],
    "Ismailia":   ["Ismailia City", "Fayed", "Abu Sultan", "El Qantara"],
    "Suez":       ["Suez City", "Ataka", "El Arbaeen", "Ain Sokhna"],
    "Minya":      ["Minya City", "Mallawi", "Abu Qurqas", "Beni Mazar"],
    "Sohag":      ["Sohag City", "Akhmim", "Girga", "Tahta"],
}

# ── Coordinates per governorate ──────────────────────────────────────────────
GOVERNORATE_COORDS = {
    "Cairo":      (30.0444, 31.2357),
    "Giza":       (30.0056, 31.2089),
    "Alexandria": (31.2001, 29.9187),
    "Qalyubia":   (30.3292, 31.2168),
    "Gharbia":    (30.8753, 31.0364),
    "Dakahlia":   (31.0364, 31.3807),
    "Sharqia":    (30.5877, 31.4983),
    "Aswan":      (24.0889, 32.8998),
    "Luxor":      (25.6872, 32.6396),
    "Red Sea":    (27.2579, 33.8116),
    "Ismailia":   (30.5965, 32.2715),
    "Suez":       (29.9668, 32.5498),
    "Minya":      (28.1099, 30.7503),
    "Sohag":      (26.5590, 31.6957),
}

INFORMAL_PHRASES = [
    "Please help quickly!", "This is a total mess.", "No one is answering the phone.",
    "It has been like this for two days.", "The whole street is blocked.",
    "Everything is broken.", "Fix it ASAP, we can't live like this.",
    "I already called before and nothing happened.", "We are suffering badly.",
]

# --- 2. Issue Content Templates ---

ISSUE_CONTENT = {
    "electricity": {
        "headlines": ["Power Outage", "Electric Sparking", "Broken Street Light",
                      "Transformer Explosion", "Frequent Voltage Drops"],
        "details": [
            "The electricity has been out at {address} for hours.",
            "Live wires are exposed on the pole near {street_name}.",
            "Power surges are damaging appliances in the whole building.",
            "A technical failure in the 500kVA transformer near {address}.",
            "Repeated outages every night since last week at {street_name}.",
        ]
    },
    "water": {
        "headlines": ["Burst Pipe", "Water Leak", "Sewage Overflow", "No Water Supply",
                      "Contaminated Water"],
        "details": [
            "A 4-inch diameter pipe burst in front of {address}.",
            "The water smells terrible and is brown in color.",
            "Sewage water is flooding the entrance of {street_name}.",
            "Main supply line is leaking heavily near the valve.",
            "Water pressure is extremely low, nothing comes from the tap.",
        ]
    },
    "gas_issue": {
        "headlines": ["Gas Leak", "Gas Smell", "Valve Damage", "Pressure Drop",
                      "Suspected Gas Explosion"],
        "details": [
            "A very strong smell of gas detected at {address}.",
            "The main gas meter is making a loud whistling sound.",
            "Low gas pressure, the heater is not working.",
            "Construction work may have damaged a pipe on {street_name}.",
            "Neighbors are evacuating due to the strong gas odor.",
        ]
    },
    "waste_garbage": {
        "headlines": ["Garbage Pile", "Dead Animal", "Illegal Dumping", "Trash Accumulation",
                      "Health Hazard Waste"],
        "details": [
            "Trash has not been collected from {street_name} for a week.",
            "Illegal dumping of construction waste near {address}.",
            "The garbage smell is unbearable and attracting insects.",
            "Dead carcass in the middle of the road causing a health risk.",
            "Mountains of waste blocking the pavement on {street_name}.",
        ]
    },
    "road_damage": {
        "headlines": ["Large Pothole", "Road Collapse", "Damaged Pavement", "Flooded Road"],
        "details": [
            "A large pothole on {street_name} is damaging vehicles.",
            "Part of the road near {address} has completely collapsed.",
            "The pavement is cracked and very dangerous for pedestrians.",
            "Road is flooded and completely impassable near {address}.",
        ]
    },
    "telecom": {
        "headlines": ["No Internet", "Phone Line Dead", "Cable Cut", "Signal Loss"],
        "details": [
            "The entire building at {address} has no internet since yesterday.",
            "The telephone landline has been dead for three days.",
            "A cable appears to have been cut near {street_name}.",
            "Mobile signal is completely lost in the {street_name} area.",
        ]
    },
}

# ── Out-of-scope topics (genuinely outside utility mandate) ─────────────────
OUT_OF_SCOPE_TEMPLATES = [
    ("Traffic Accident Report",        "A car accident occurred at {street_name}, need police assistance."),
    ("Noise Complaint",                "Neighbors are making too much noise late at night at {address}."),
    ("School Enrollment Question",     "I want to know how to enroll my child in the nearby school."),
    ("Lost Pet",                       "My cat has been missing since yesterday near {address}."),
    ("Request for Financial Aid",      "I need financial assistance, I cannot afford rent this month."),
    ("Political Complaint",            "I want to file a complaint about the local council decisions."),
    ("Medical Emergency",              "Someone collapsed on {street_name}, we need an ambulance now."),
    ("Construction Permit Question",   "How do I get a permit to renovate my flat at {address}?"),
    ("Neighborhood Security Issue",    "There are suspicious people gathering near {street_name} at night."),
    ("Marriage Certificate Request",   "Where can I get a copy of my marriage certificate?"),
]

# ── False report signals (plausible but fabricated) ─────────────────────────
FALSE_SIGNAL_PREFIXES = [
    "I think there might be",
    "Someone told me there is",
    "I heard rumors about",
    "I am not sure but maybe",
    "A friend said he saw",
]

FALSE_CONTRADICTIONS = [
    "but I have not seen it myself.",
    "though everything looks fine from outside.",
    "but the workers said it was already fixed.",
    "I could be wrong about this.",
    "although my neighbor disagrees.",
]

# ── Repeated report signals ──────────────────────────────────────────────────
REPEATED_PREFIXES = [
    "Following up on my previous report about",
    "I reported this before but it was not fixed:",
    "This is the second time I am reporting",
    "Still not resolved, I reported this last week:",
    "Resubmitting my complaint about",
]

EMERGENCY_DESCRIPTIONS = [
    "The building is shaking and cracking.",
    "I see fire and thick smoke.",
    "People are suffocating and cannot breathe.",
    "The situation is critical and life-threatening.",
    "Immediate danger — residents are evacuating.",
    "Children and elderly people are trapped inside.",
    "The structure looks like it may collapse.",
]

# --- 3. Helper Functions ---

def get_realistic_coordinates(governorate):
    """Use full coordinate table — no more (30.0, 31.0) default fallback."""
    lat_c, lon_c = GOVERNORATE_COORDS.get(governorate, (26.8206, 30.8025))  # Egypt centroid
    return (
        round(lat_c + random.uniform(-0.08, 0.08), 6),
        round(lon_c + random.uniform(-0.08, 0.08), 6),
    )

def _placeholders():
    return {
        "street_name": f"Street {random.randint(1, 150)}",
        "address":     f"Building {random.randint(1, 80)}",
    }

def generate_report_details(issue_type, report_truth):
    ph = _placeholders()

    # ── Out_of_scope: completely different domain ────────────────────────────
    if report_truth == "Out_of_scope":
        headline_tpl, detail_tpl = random.choice(OUT_OF_SCOPE_TEMPLATES)
        return headline_tpl.format(**ph), detail_tpl.format(**ph)

    content  = ISSUE_CONTENT[issue_type]
    headline = random.choice(content["headlines"])
    detail   = random.choice(content["details"])

    # ── True: realistic report + optional informal noise ────────────────────
    if report_truth == "True":
        if random.random() < 0.4:
            detail = f"{detail} {random.choice(INFORMAL_PHRASES)}"

    # ── False: plausible but hedged / second-hand ────────────────────────────
    elif report_truth == "False":
        prefix       = random.choice(FALSE_SIGNAL_PREFIXES)
        contradiction = random.choice(FALSE_CONTRADICTIONS)
        detail = f"{prefix} {detail.lower()} {contradiction}"

    # ── Repeated: explicit follow-up language ────────────────────────────────
    elif report_truth == "Repeated":
        prefix  = random.choice(REPEATED_PREFIXES)
        detail  = f"{prefix} {detail}"
        headline = f"[Follow-up] {headline}"

    # ── Emergency: dangerous context; keywords only 40% of the time ─────────
    elif report_truth == "Emergency":
        if random.random() < 0.4:
            modifier = random.choice(["URGENT", "CRITICAL", "EXPLOSION", "FIRE"])
            headline = f"!!! {modifier} !!! - {headline}"
            detail   = f"{modifier}!! {detail}"
        else:
            detail = f"{detail} {random.choice(EMERGENCY_DESCRIPTIONS)}"
        if random.random() < 0.4:
            detail = f"{detail} {random.choice(INFORMAL_PHRASES)}"

    # ── Unclear: very short / garbled text ───────────────────────────────────
    elif report_truth == "Unclear":
        unclear_headlines = ["I don't know", "test report", "...", "Check this", "asdfgh", "???"]
        unclear_details   = ["not sure", "...", "test", "asdfgh", "help", "idk"]
        headline = random.choice(unclear_headlines)
        detail   = random.choice(unclear_details) if random.random() > 0.5 else detail[:15]

    return headline.format(**ph), detail.format(**ph)

# --- 4. Main Generation Logic ---

def generate_synthetic_data(num_records=15000):
    reports = []
    truth_weights = {
        "True":         0.40,
        "Emergency":    0.15,
        "Repeated":     0.15,
        "False":        0.10,
        "Out_of_scope": 0.10,
        "Unclear":      0.10,
    }

    for _ in range(num_records):
        report_truth = random.choices(
            list(truth_weights.keys()),
            weights=list(truth_weights.values())
        )[0]

        issue_type  = random.choice(list(ISSUE_CONTENT.keys()))
        governorate = random.choice(list(EGYPT_GEOGRAPHY.keys()))
        city        = random.choice(EGYPT_GEOGRAPHY[governorate])
        lat, lon    = get_realistic_coordinates(governorate)

        headline, detail = generate_report_details(issue_type, report_truth)

        text_length = len(headline + " " + detail)

        # Proximity / repetition flags (kept for potential future use)
        if report_truth == "False":
            proximity  = random.choices([0, 1], weights=[0.85, 0.15])[0]
            repetition = 0
        elif report_truth == "Repeated":
            proximity  = 1
            repetition = 1
        else:
            proximity  = random.choices([1, 0], weights=[0.75, 0.25])[0]
            repetition = random.choices([0, 1], weights=[0.92, 0.08])[0]

        reports.append({
            "report_date":          (datetime.now() - timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d'),
            "governorate":          governorate,
            "city":                 city,
            "longitude":            lon,
            "latitude":             lat,
            "issue_type":           issue_type,
            "report_headline":      headline,
            "report_detail":        detail,
            "text_length":          text_length,
            "report_category":      random.choices(["citizen", "business"], weights=[0.9, 0.1])[0],
            "proximity_to_service": proximity,
            "repetition_flag":      repetition,
            "report_truth":         report_truth,
        })

    return pd.DataFrame(reports)


if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)

    df = generate_synthetic_data(15000)
    df.to_csv("synthetic_reports_egypt.csv", index=False)

    print("✅ Dataset generated: 'synthetic_reports_egypt.csv'")
    print(f"\n📊 Class distribution:")
    print(df['report_truth'].value_counts().to_string())
    print(f"\n📝 Sample entries per class:")
    for label in df['report_truth'].unique():
        row = df[df['report_truth'] == label].iloc[0]
        print(f"\n[{label}]")
        print(f"  Headline : {row['report_headline']}")
        print(f"  Detail   : {row['report_detail']}")

✅ Dataset generated: 'synthetic_reports_egypt.csv'

📊 Class distribution:
report_truth
True            5946
Repeated        2306
Emergency       2303
Unclear         1510
Out_of_scope    1469
False           1466

📝 Sample entries per class:

[Repeated]
  Headline : [Follow-up] Power Outage
  Detail   : Still not resolved, I reported this last week: Repeated outages every night since last week at Street 27.

[True]
  Headline : Road Collapse
  Detail   : Road is flooded and completely impassable near Building 54.

[False]
  Headline : Phone Line Dead
  Detail   : I heard rumors about the entire building at Building 34 has no internet since yesterday. although my neighbor disagrees.

[Emergency]
  Headline : Contaminated Water
  Detail   : Main supply line is leaking heavily near the valve. Immediate danger — residents are evacuating.

[Unclear]
  Headline : ???
  Detail   : ...

[Out_of_scope]
  Headline : Request for Financial Aid
  Detail   : I need financial assistance, I cannot aff